# Test — concept subspace creation

Validates the concept-subspace *creation* path of the `subspace/` pipeline
against a real model: building `ConceptSubspace` objects from WordNet synsets and
checking their basis is well-formed (orthonormal, fixed-rank, disk-cached).

The CPU-only math is covered by `tests/test_subspace.py`; this notebook is the
model-in-the-loop counterpart.

In [ ]:
import sys
from pathlib import Path

# repo root is two levels up from subspace/notebooks/, so `subspace`/`frames` import
sys.path.insert(0, str(Path.cwd().parents[1]))

In [ ]:
import torch

from subspace import SubspaceUnembeddingRepresentation

In [ ]:
RANK = 8
MIN_LEMMAS = 11
MAX_TOKENS = 3

sur = SubspaceUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    rank=RANK,
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

## Create concept subspaces

In [ ]:
dog = sur.get_concept("dog.n.01", MIN_LEMMAS, MAX_TOKENS)
war = sur.get_concept_cached("war.n.01", MIN_LEMMAS, MAX_TOKENS)
print(dog)
print(war)

## Validate the created basis

In [ ]:
d = sur.unembedding.unembedding_matrix.shape[-1]

# fixed-rank shape
assert dog.basis.shape == (d, RANK), dog.basis.shape
assert dog.rank == RANK

# columns orthonormal up to the kept rank
col_norms = dog.basis.norm(dim=0)
kept = col_norms > 0.5
b = dog.basis[:, kept].float()
gram = b.mT @ b
ortho = torch.allclose(gram, torch.eye(int(kept.sum()), device=gram.device), atol=1e-3)
print(f"kept directions: {int(kept.sum())}/{RANK}")
print("per-column norms:", col_norms.tolist())
print("BᵀB ≈ I on kept columns:", ortho)
assert ortho

In [ ]:
# cache round-trip: get_concept_cached must return an identical basis and write to disk
path = sur._concept_cache_path("war.n.01", MIN_LEMMAS, MAX_TOKENS)
war2 = sur.get_concept_cached("war.n.01", MIN_LEMMAS, MAX_TOKENS)
print("cache file:", path)
print("exists:", path.exists())
print("identical basis on reload:", torch.equal(war.basis, war2.basis))
assert path.exists()
assert torch.equal(war.basis, war2.basis)

## Sanity: the created subspace is meaningful

In [ ]:
relevant = "The puppy barked and wagged its tail at the hound"
unrelated = "The committee reviewed the quarterly budget report"

rel = sur.expression(relevant, dog).item()
unrel = sur.expression(unrelated, dog).item()
print(f"expression(relevant, dog)  = {rel:.4f}")
print(f"expression(unrelated, dog) = {unrel:.4f}")
assert rel > unrel